# Fake News Detector

In [1]:
# Basic imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
# Pre-processing
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
# Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
# Model
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Data Loading 
* Load and examine dataset
* Convert datatypes
* Treat missing values

In [2]:
df = pd.read_csv("Dataset.csv")

In [3]:
# Examining Dataset 
print("Dataset shape:",df.shape)
print("\nSample: ")
df.head(1)

Dataset shape: (72134, 4)

Sample: 


,Unnamed: 0,title,text,label
0,0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,No comment is expected from Barack Obama Membe...,1


In [4]:
# Checking and converting datatypes
print("Original Datatypes:\n",df.dtypes)
df['title'] = df['title'].astype('string')
df['text'] = df['text'].astype('string')
print("\nChanged Datatypes:\n",df.dtypes)

Original Datatypes:
 Unnamed: 0     int64
title         object
text          object
label          int64
dtype: object

Changed Datatypes:
 Unnamed: 0             int64
title         string[python]
text          string[python]
label                  int64
dtype: object


In [5]:
# Treating missing values
print("Number of missing values per column:\n",df.isna().sum())
df['title'] = df['title'].fillna('')
df['text'] = df['text'].fillna('')
print("\nAfter treating missing values-")
print("Number of missing values per column:\n",df.isna().sum())

Number of missing values per column:
 Unnamed: 0      0
title         558
text           39
label           0
dtype: int64

After treating missing values-
Number of missing values per column:
 Unnamed: 0    0
title         0
text          0
label         0
dtype: int64


## Preprocessing

In [6]:
def preprocess(df, remove_stopwords=True, remove_punctuation=True, lowercase=True, check= True):
    df = df.copy()
    # Convert to Lowercase
    if lowercase:
        df['text'] = df['text'].str.lower()
        df['title'] = df['title'].str.lower()

    # Remove punctuation
    if remove_punctuation: 
        df['title'] = df['title'].str.replace(r'[^\w\s]+', '', regex=True) 
        df['text'] = df['text'].str.replace(r'[^\w\s]+', '', regex=True) 

    # Remove stopwords (Conditional)
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))

        df['text'] = df['text'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )
        df['title'] = df['title'].apply(
            lambda x: " ".join([word for word in x.split() if word not in stop_words])
        )

    # Check if pre-processing caused too many empty rows
    if check: 
        print("No of empty title rows: ",(df['title'].str.len() == 0).sum())
        print("No of empty text rows: ",(df['text'].str.len() == 0).sum())
        count = ((df['title'].str.len() == 0) & (df['text'].str.len() == 0)).sum()
        print("No of rows with empty title AND text: ",count)
    
    return df

## Vectorization 

In [7]:
def vectorize(df,mdf=5, ngramRange=(1,2), max_features_title=50000, max_features_text=500000, verbose = False):
    X = df[["title", "text"]]
    Y = df["label"]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.33, random_state=42)

    # Initialize the TF-IDF Vectorizer
    vectorizer_title = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_title)
    vectorizer_text = TfidfVectorizer(min_df=mdf, ngram_range=ngramRange, max_features=max_features_text)

    # Fit and transform the data
    tfidf_title_train = vectorizer_title.fit_transform(X_train["title"])
    tfidf_text_train = vectorizer_text.fit_transform(X_train["text"])

    tfidf_title_test = vectorizer_title.transform(X_test["title"])
    tfidf_text_test = vectorizer_text.transform(X_test["text"])

    # Matrix inspection
    # Shape = (number_of_documents, number_of_features)
    if verbose:
        print(tfidf_title_train.shape)
        print(tfidf_text_train.shape)

    X_train = hstack([tfidf_title_train, tfidf_text_train])
    X_test = hstack([tfidf_title_test, tfidf_text_test])

    feature_names = np.concatenate((
    vectorizer_title.get_feature_names_out(),
    vectorizer_text.get_feature_names_out()
    ))

    total_features = X_train.shape[1]
    return X_train, X_test, Y_train, Y_test, feature_names, total_features

## Evaluation

In [8]:
def evaluate(Y_test, Y_pred, model, feature_names, plot_cm= False):
    # Metrics
    print("Accuracy: ", accuracy_score(Y_test, Y_pred))
    print("Precision: ", precision_score(Y_test, Y_pred))
    print("Recall: ", recall_score(Y_test, Y_pred))
    print("F1: ", f1_score(Y_test, Y_pred, average="macro"))

    # Confusion Matrix
    cm = confusion_matrix(Y_test, Y_pred)
    print("\nConfusion Matrix:\n", cm)

    if plot_cm:
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Class 0', 'Class 1'])
        disp.plot(cmap=plt.cm.Blues)
        plt.show()

    # Top positive and negative features
    coefficients = model.coef_[0] 

    df_features = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients
    })

    top_negative_features = df_features.sort_values(by='Coefficient', ascending=True).head(5)
    top_positive_features = df_features.sort_values(by='Coefficient', ascending=False).head(5)

    print("\nTop 5 Negative Features:")
    print(top_negative_features)

    print("\nTop 5 Positive Features:")
    print(top_positive_features)

## Models

### Model 1
Model - Logistic Regression     

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 44.1s   
* Training Time : 2.5s                

Metrics -           
* Accuracy:  0.96782
* Precision:  0.95957
* Recall:  0.97820
* F1:  0.96779
* Confusion Matrix:         
 [11148     501        -> False Positives            
   265      11891]     -> False Negatives              

In [9]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df)

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"Features created in {creation_time:.4f} seconds.")

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1
Features created in 44.1255 seconds.


In [10]:
# Training model
start_time = time.perf_counter()

model = LogisticRegression(random_state=42).fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 2.5602 seconds.


In [11]:
# Evaluation
evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9678218861583701
Precision:  0.9595706907682375
Recall:  0.9782000658111221
F1:  0.9677905081677439

Confusion Matrix:
 [[11148   501]
 [  265 11891]]

Top 5 Negative Features:
                   Feature  Coefficient
352973             reuters   -19.644100
361547                said   -18.528369
1810             breitbart   -14.876882
432972              trumps    -9.132123
454520  washington reuters    -7.019568

Top 5 Positive Features:
         Feature  Coefficient
15125      video    10.903366
446457       via     8.507856
1777    breaking     6.366823
205873     image     6.172324
197585   hillary     5.396140


### Model Analysis

#### Feature Importance Observations

* The most influential features for classification were:

  * **Negative coefficients (Real News indicators):** `reuters`, `said`, `washington reuters`, `new york`, `york times`, `president donald`, `twitter`
  * **Positive coefficients (Fake News indicators):** `video`, `via`, `breaking`, `image`, `image via`, `hillary`, `2016`, `october`, `trump`, `november`

#### Interpretation

* The model appears to heavily rely on **source-related terms** such as `reuters`, `washington reuters`, and `breitbart`.
* Several highly weighted features correspond to **publisher names, locations, or news organizations**, suggesting that source identification may contribute significantly to classification performance.
* Features associated with fake news tend to include **media-sharing and clickbait-style terms** such as `video`, `image`, `via`, and `breaking`.
* The large coefficient magnitudes indicate that some features are highly predictive of a particular class.

#### N-gram Analysis

* Although bigrams appeared among the top features (e.g., `washington reuters`, `image via`, `new york`), the majority of the most influential features were still unigrams.
* This suggests that most predictive power may come from individual words, while bigrams provide additional contextual information.
* Further comparison with a unigram-only model is required to quantify the actual contribution of bigrams.

#### Model Behavior

* The model achieved high and balanced performance:

  * Accuracy: 96.78%
  * Precision: 95.96%
  * Recall: 97.82%
  * F1 Score: 96.78%
* Recall is slightly higher than precision, indicating that the model misses relatively few positive samples but produces somewhat more false positives.
* The confusion matrix shows that classification errors are relatively low compared to the total number of samples.

#### Computational Performance

* Training time for Logistic Regression was approximately **2.5 seconds**.
* TF-IDF vectorization (~37 seconds) was significantly more computationally expensive than model training.
* For this pipeline, feature extraction is the primary computational bottleneck rather than classifier training.

### Model 2